# 05 - Deep Agent com skills

Uma **skill** é um pacote de instruções para uma tarefa específica. Cada skill é uma pasta com um arquivo `SKILL.md`, que tem um cabeçalho com `name` e `description`, seguido do passo a passo.

Como o agente usa skills (divulgação progressiva):

1. Ao iniciar, ele vê só a **lista** de skills: nome, descrição e caminho do arquivo.
2. Quando a tarefa combina com uma descrição, ele lê o `SKILL.md` com `read_file`.
3. Só então segue as instruções. Assim, o contexto não fica cheio com instruções que não serão usadas.

Este notebook usa duas skills, em `skills/`:

- `resumo-vendas`: como montar um resumo de vendas por produto;
- `revisao-contas`: como conferir os números antes de entregar.

O agente também usa planejamento (`write_todos`) e as ferramentas `consultar_vendas` e `calcular`, como nos notebooks anteriores.

Fluxo:

```
START → lê as skills → planeja (write_todos) → dados (consultar_vendas) → contas (calcular) → revisa (skill revisao-contas) → resposta final → END
```

Tudo fica registrado no MLflow, no experimento `deep-agent-skills`.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook. O `mlflow.langchain.autolog()` registra cada execução, incluindo as leituras de skill.

In [1]:
import ast
import json
import operator
import os
import unicodedata
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-skills"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. Skills

Cada pasta em `skills/` com um `SKILL.md` é uma skill. Abaixo, as skills encontradas e a descrição de cada uma, lida do cabeçalho.

As skills são carregadas no estado do agente, em `/skills/<nome>/SKILL.md`. Isso é o backend padrão (`StateBackend`): o agente lê as skills pelo estado, sem acessar o disco.

In [2]:
PASTA_SKILLS = Path("skills")

ARQUIVOS_SKILLS = {}
for arquivo in sorted(PASTA_SKILLS.glob("*/SKILL.md")):
    texto = arquivo.read_text(encoding="utf-8")
    ARQUIVOS_SKILLS[f"/skills/{arquivo.parent.name}/SKILL.md"] = create_file_data(texto)

for caminho, dados in ARQUIVOS_SKILLS.items():
    texto = dados["content"]
    descricao = next(
        linha.split(":", 1)[1].strip()
        for linha in texto.splitlines()
        if linha.startswith("description:")
    )
    print(f"{caminho}\n  {descricao}\n")

/skills/resumo-vendas/SKILL.md
  Como montar um resumo de vendas de um período, com total por produto e o produto mais vendido. Use quando a tarefa pedir relatório, resumo ou ranking de vendas.

/skills/revisao-contas/SKILL.md
  Como conferir contas e totais antes de entregar um resultado. Use sempre que a resposta trouxer somas, médias ou comparações numéricas.



## 3. Ferramentas

- `consultar_vendas`: devolve as vendas de um mês. Os dados são simulados.
- `calcular`: faz contas com números e as quatro operações, usando `ast`. Nunca use `eval` com texto gerado por LLM.

In [3]:
VENDAS_API = {
    "janeiro": {"cafe": 120, "pao": 300, "queijo": 45},
    "fevereiro": {"cafe": 135, "pao": 280, "queijo": 50},
    "marco": {"cafe": 150, "pao": 310, "queijo": 38},
}


@tool
def consultar_vendas(mes: str) -> str:
    """Consulta as vendas de um mês do primeiro trimestre. Retorna a quantidade vendida de cada produto em JSON. Meses válidos: janeiro, fevereiro, março."""
    chave = unicodedata.normalize("NFKD", mes.lower().strip()).encode("ascii", "ignore").decode()
    dados = VENDAS_API.get(chave)
    if dados is None:
        return json.dumps({"mes": mes, "erro": "mês não encontrado"}, ensure_ascii=False)
    return json.dumps({"mes": mes, "vendas": dados}, ensure_ascii=False)


OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '(120 + 135) * 2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


FERRAMENTAS = [consultar_vendas, calcular]

print(consultar_vendas.name, "|", calcular.name)

consultar_vendas | calcular


## 4. Criando o agente

- `skills=["/skills/"]` diz ao agente onde estão as skills no estado.
- `TodoListMiddleware` adiciona o `write_todos`, para o plano.
- O `system_prompt` manda ler a skill certa antes de agir. Ele não descreve os passos: quem descreve são os `SKILL.md`.

In [4]:
agente = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    skills=["/skills/"],
    system_prompt=(
        "Você é um analista de vendas. Em tarefas com etapas, a primeira ação é chamar write_todos "
        "com um plano de 3 a 5 passos. Antes de executar, leia com read_file o SKILL.md das skills "
        "que combinam com a tarefa e siga as instruções delas. Responda em português."
    ),
)

## 5. Executando

A função `executar` roda a tarefa e mostra as mensagens na ordem em que aconteceram:

- `Plano` é a lista de passos da primeira chamada de `write_todos`. Chamadas seguintes atualizam o status.
- `→` é uma chamada de ferramenta. Leituras de `SKILL.md` aparecem como `read_file`.
- `←` é o resultado devolvido.
- `Resposta` é a mensagem final.

In [5]:
def resumir(texto: str, limite: int = 200) -> str:
    return texto if len(texto) <= limite else texto[:limite] + "..."


def executar(tarefa: str) -> dict:
    estado = agente.invoke({
        "messages": [HumanMessage(content=tarefa)],
        "files": ARQUIVOS_SKILLS,
    })

    linhas = [f"**Tarefa:** {tarefa}", ""]
    plano_mostrado = False
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                if chamada["name"] == "write_todos" and not plano_mostrado:
                    linhas.append("**Plano:**")
                    for item in chamada["args"].get("todos", []):
                        linhas.append(f"- {item.get('content', item)}")
                    linhas.append("")
                    plano_mostrado = True
                elif chamada["name"] == "write_todos":
                    linhas.append("→ `write_todos` (atualiza o status dos passos)")
                else:
                    linhas.append(f"→ `{chamada['name']}({resumir(str(chamada['args']))})`")
        elif isinstance(mensagem, ToolMessage) and mensagem.name != "write_todos":
            linhas.append(f"← `{resumir(str(mensagem.content))}`")
        elif isinstance(mensagem, AIMessage) and mensagem.content:
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))
    return estado


estado = executar(
    "Faça o resumo de vendas do primeiro trimestre, com o total de cada produto e o mais vendido. "
    "Confira as contas antes de responder."
)

**Tarefa:** Faça o resumo de vendas do primeiro trimestre, com o total de cada produto e o mais vendido. Confira as contas antes de responder.

**Plano:**
- Ler SKILL.md de 'resumo-vendas' e 'revisao-contas' e seguir instruções
- Consultar vendas de janeiro, fevereiro e março
- Calcular totais por produto e identificar o produto mais vendido
- Revisar as contas usando a skill 'revisao-contas' e preparar o resumo final

→ `read_file({'file_path': '/skills/resumo-vendas/SKILL.md', 'limit': 1000})`
← `@@ lines 1-18 of 18 @@
---
name: resumo-vendas
description: Como montar um resumo de vendas de um período, com total por produto e o produto mais vendido. Use quando a tarefa pedir relatório, resumo o...`
→ `read_file({'file_path': '/skills/revisao-contas/SKILL.md', 'limit': 1000})`
← `@@ lines 1-15 of 15 @@
---
name: revisao-contas
description: Como conferir contas e totais antes de entregar um resultado. Use sempre que a resposta trouxer somas, médias ou comparações numéricas.
---...`
→ `consultar_vendas({'mes': 'janeiro'})`
→ `consultar_vendas({'mes': 'fevereiro'})`
→ `consultar_vendas({'mes': 'março'})`
← `{"mes": "janeiro", "vendas": {"cafe": 120, "pao": 300, "queijo": 45}}`
← `{"mes": "fevereiro", "vendas": {"cafe": 135, "pao": 280, "queijo": 50}}`
← `{"mes": "março", "vendas": {"cafe": 150, "pao": 310, "queijo": 38}}`
→ `write_todos` (atualiza o status dos passos)
→ `calcular({'expressao': '120 + 135 + 150'})`
← `405`
→ `calcular({'expressao': '300 + 280 + 310'})`
← `890`
→ `calcular({'expressao': '45 + 50 + 38'})`
← `133`
→ `write_todos` (atualiza o status dos passos)
→ `calcular({'expressao': '405 + 890 + 133'})`
← `1428`
→ `write_todos` (atualiza o status dos passos)

**Resposta:** Resumo de vendas (1º trimestre)

Tabela — total por produto (unidades no período:
- café: 405
- pão: 890
- queijo: 133

Produto mais vendido: pão — 890 unidades

Valores mensais usados:
- Janeiro: café 120, pão 300, queijo 45
- Fevereiro: café 135, pão 280, queijo 50
- Março: café 150, pão 310, queijo 38

Contas conferidas.

## 6. Conferindo o uso das skills

Listamos as leituras de arquivo e a ordem das chamadas. O esperado: o agente lê os dois `SKILL.md` antes de consultar os dados, e `write_todos` aparece antes das ferramentas de dados.

In [6]:
ordem = []
lidos = []
for mensagem in estado["messages"]:
    if isinstance(mensagem, AIMessage):
        for chamada in mensagem.tool_calls:
            ordem.append(chamada["name"])
            if chamada["name"] == "read_file":
                lidos.append(chamada["args"].get("file_path"))

print(" → ".join(ordem))
print()
print("Skills lidas:", [caminho for caminho in lidos if caminho and "SKILL.md" in caminho])
print("Planejou primeiro:", ordem[0] == "write_todos")
print("Leu skill antes dos dados:", bool(lidos) and ordem.index("read_file") < ordem.index("consultar_vendas"))

write_todos → read_file → read_file → consultar_vendas → consultar_vendas → consultar_vendas → write_todos → calcular → calcular → calcular → write_todos → calcular → write_todos

Skills lidas: ['/skills/resumo-vendas/SKILL.md', '/skills/revisao-contas/SKILL.md']
Planejou primeiro: True
Leu skill antes dos dados: True


## 7. Acompanhando o rastreamento no MLflow

Cada execução virou um trace no experimento `deep-agent-skills`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-skills` e abra a aba **Traces**.
5. Clique no trace mais recente. Os spans `TOOL` mostram `read_file` para cada `SKILL.md`, além de `write_todos`, `consultar_vendas` e `calcular`. Dá para ver o conteúdo lido em cada span.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista os spans pelo notebook, sem abrir a interface.

In [7]:
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    if span.span_type == "TOOL":
        print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 1

Spans do trace mais recente (tr-ced4a69bafe016561cdd0c24874afbaf):
- write_todos (TOOL)
- read_file (TOOL)
- read_file (TOOL)
- consultar_vendas (TOOL)
- consultar_vendas (TOOL)
- consultar_vendas (TOOL)
- write_todos (TOOL)
- calcular (TOOL)
- calcular (TOOL)
- calcular (TOOL)
- write_todos (TOOL)
- calcular (TOOL)
- write_todos (TOOL)


/tmp/ipykernel_58374/2815719652.py:1: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## Observações

- Skills são instruções que o agente lê sob demanda. Elas não são código: não executam nada sozinhas.
- A `description` é o que o agente usa para decidir se a skill combina com a tarefa. Descrições vagas levam a leituras erradas ou a nenhuma leitura.
- O `system_prompt` diz que as skills devem ser lidas, mas não repete o conteúdo delas. Assim, trocar uma skill não exige mexer no agente.
- Com o `StateBackend`, as skills são passadas no `invoke`. Com o `FilesystemBackend`, o agente lê as skills direto do disco.
- O resultado pode variar entre execuções. Compare as leituras de skill e as chamadas com a resposta final.